# Feature + Evaluation Lab — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*

Combine feature engineering + evaluation discipline on an imbalanced, cost-sensitive classification problem.

**Outcome.** Students can:
- build a leakage-safe sklearn pipeline,
- compare baselines vs stronger models under CV,
- use PR-AUC / ROC-AUC appropriately,
- choose a decision threshold using business costs,
- run slice analysis and error analysis.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
)

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Feature + Evaluation Lab — Credit Default (Student)

This is a capstone-style classical ML lab: baselines -> CV -> thresholding -> slices.

## Section 0 — Load dataset (with fallback)

Expected path: `data/credit_default/credit.csv`

Supported target column names (0/1):
- default / Default
- default.payment.next.month
- DEFAULT_PAY
- default_payment_next_month

In [ ]:
$10

**Why this works.** # Feature + Evaluation Lab — Credit Default (Instructor)

## Setup — Load dataset (with fallback)

## Task 2: Identify target column robustly
*Section: Define X/y + holdout split*

## Section 1 — Define X/y + holdout split

### Task 1.1: Identify target column robustly

- Find the target col from the supported names
- Create X/y
- Split into train/holdout (stratified)

**Checkpoint:** Why do we keep a final holdout set untouched during model selection?

In [ ]:
target_candidates = ['default', 'Default', 'default.payment.next.month', 'DEFAULT_PAY', 'default_payment_next_month']
target_col = next((c for c in target_candidates if c in df.columns), None)
if target_col is None:
    raise ValueError('Target column not found. Expected one of: ' + ', '.join(target_candidates))
y = df[target_col].astype(int)
X = df.drop(columns=[target_col])

Xtr, Xho, ytr, yho = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print('base_rate_train', float(ytr.mean()))


In [ ]:
# Checks
check('target_found', target_col is not None)

**Why this works.** ## Section 1 — Define X/y + holdout split

### Task 1.1: Identify target column robustly

## Task 3: Majority baseline
*Section: Baselines*

## Section 2 — Baselines

### Task 2.1: Majority baseline

In [ ]:
# Majority baseline
majority = int(ytr.mean() >= 0.5)
pred = np.full(len(yho), majority)
print('majority_acc', accuracy_score(yho, pred))


**Why this works.** ## Section 2 — Baselines

### Task 2.1: Majority baseline

## Task 4: Baseline pipeline (LogisticRegression)
*Section: Baselines*

### Task 2.2: Baseline pipeline (LogisticRegression)

- Build preprocess (numeric: impute+scale, categorical: impute+onehot)
- Evaluate with CV on training set (ROC-AUC + PR-AUC)

In [ ]:
$11

**Why this works.** under imbalance, PR-AUC often matches product utility better than ROC-AUC.

## Task 5: Compare LR vs RF vs GB
*Section: Model comparison (CV)*

## Section 3 — Model comparison (CV)

### Task 3.1: Compare LR vs RF vs GB

- Compare under the same CV splits
- Report ROC-AUC and PR-AUC

**FAANG Gotcha:** ROC-AUC can look good even when precision is poor under imbalance.

In [ ]:
models = {
    'lr': LogisticRegression(max_iter=2000),
    'rf': RandomForestClassifier(n_estimators=100, random_state=0),
    'gb': GradientBoostingClassifier(n_estimators=120, random_state=0),
}

cv_summary = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    scores = cross_validate(pipe, Xtr, ytr, cv=cv, scoring=scoring, return_train_score=False)
    cv_summary.append((name, float(scores['test_roc_auc'].mean()), float(scores['test_pr_auc'].mean())))

print('CV summary (name, roc_auc_mean, pr_auc_mean)')
for row in cv_summary:
    print(row)

# Rationale: compare models under identical CV to avoid unfair selection.


**Why this works.** compare models under identical CV to avoid unfair selection.

## Section 3 — Model comparison (CV)

### Task 3.1: Compare LR vs RF vs GB

### Figure: CV Model Comparison

![CV Model Comparison](/images/w5-d5-model-comparison.png)

## Task 6: Add at least one engineered feature
*Section: Feature engineering (lightweight)*

## Section 4 — Feature engineering (lightweight)

### Task 4.1: Add at least one engineered feature

Examples (only if columns exist):
- ratios like PAY_AMT1 / BILL_AMT1 (utilization proxy)
- log1p on skewed amount columns

In [ ]:
$12

**Why this works.** engineered ratios can capture behavioral signals; always validate under CV.

## Section 4 — Feature engineering (lightweight)

### Task 4.1: Add at least one engineered feature

## Task 7: Fit best pipeline on Xtr_fe and evaluate on holdout
*Section: Final fit + cost-based threshold*

## Section 5 — Final fit + cost-based threshold

### Task 5.1: Fit best pipeline on Xtr_fe and evaluate on holdout

Then pick a threshold using costs (FN cost 5x FP).

In [ ]:
$14

**Why this works.** thresholding is a product decision; costs drive the operating point.

## Section 5 — Final fit + cost-based threshold

### Task 5.1: Fit best pipeline on Xtr_fe and evaluate on holdout

### Figure: Threshold vs Cost

![Threshold Cost Curve](/images/w5-d5-threshold-cost.png)

### Figure: Holdout Confusion Matrix

![Holdout Confusion Matrix](/images/w5-d5-confusion-matrix.png)

## Task 8: FP/FN examples
*Section: Error analysis + slices*

## Section 6 — Error analysis + slices

### Task 6.1: FP/FN examples
### Task 6.2: Slice analysis (choose a column)

- Build a holdout dataframe with y_true, p, y_pred
- Show a few FP and FN
- Slice by a bucketed numeric feature (e.g., age bucket / credit limit bucket) if columns exist

In [ ]:
# Error analysis + slice example (if columns exist)
ho = Xho_fe.copy()
ho['y_true'] = yho.values
ho['p'] = p_ho
ho['y_pred'] = yhat_ho

fp = ho[(ho['y_true'] == 0) & (ho['y_pred'] == 1)]
fn = ho[(ho['y_true'] == 1) & (ho['y_pred'] == 0)]
print('fp', len(fp), 'fn', len(fn))

for col in ['AGE', 'LIMIT_BAL', 'age', 'limit_bal']:
    if col in ho.columns:
        bins = pd.qcut(pd.to_numeric(ho[col], errors='coerce'), q=4, duplicates='drop')
        tmp = ho.copy()
        tmp['bucket'] = bins
        print('slice by', col)
        print(tmp.groupby('bucket').apply(lambda d: pd.Series({
            'n': len(d),
            'default_rate': float(np.mean(d['y_true'])),
            'acc': float(np.mean(d['y_true'].values == d['y_pred'].values)),
        })))
        break

# Rationale: slice analysis reveals where the model fails and guides targeted feature work.


**Why this works.** slice analysis reveals where the model fails and guides targeted feature work.

## Section 6 — Error analysis + slices

### Task 6.1: FP/FN examples

### Figure: FP vs FN Counts

![FP vs FN Counts](/images/w5-d5-fp-fn-counts.png)